# Fixed 16 MW facility — HPP sizing across reliability targets

This is a **separate copy** of the joint HPP–DC notebook. It uses the same unchanged HyDesign model, maximum-availability CPLEX EMS, native Parallel EGO, economics, and rational reliability penalty. Only **HPP capacities** are optimized; the facility demand is fixed to **16 MW** (installed IT capacity = 16 / PUE MW). One run per reliability target is saved in a separate batch folder with a combined `summary.csv`, a dedicated `evaluations/` folder, and complete native run subfolders.

**Objective note:** this notebook still maximizes annual integrated system value minus the reliability penalty. It does **not** switch the model to pure minimum-HPP-cost sizing. The same IT revenue rate applies to every reliability case.


In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
from uuid import uuid4
import json
import math
import shutil
import traceback
import pandas as pd
from IPython.display import display, Image
from hydesign.sizing_native_joint_hpp_dc_sizing import (
    ROOT, read_configuration, run_native_joint_hpp_dc_sizing, load_joint_hpp_dc_sizing)
from hydesign.assembly.hpp_assembly_simple_datacenter_joint_hpp_dc_sizing import IT_NAME

# ----------------------------- EDIT THESE SETTINGS -----------------------------
facility_load_fixed_mw = 16.0       # 16 MW TOTAL FACILITY demand (not IT-only)
pue = 1.15
revenue_rate_eur_per_mwh_it = 4000  # Hold fixed across the reliability sweep
reliability_targets = [0.90, 0.95, 0.99, 0.995, 0.999]
outer_time_budget = 90 * 60        # 90 minutes PER reliability target (7.5 h total)

# The previous YAML lower bounds (250 MW wind, 200 MW PV, 50 MW BESS)
# force an oversized HPP for a fixed 16 MW load. Lower them only here.
# The UPPER limits still come directly from the current local YAML.
minimum_hpp_bounds = {
    'Nwt': 5,                       # 5 x 5 MW = 25 MW wind minimum
    'solar_MW [MW AC]': 10.0,
    'b_P [MW]': 5.0,
    'b_E_h [h]': 2.0,
}

saved_run = None                   # Optional: an existing native run directory, to reload
regenerate_saved_figures = False
# -------------------------------------------------------------------------------

config_path = ROOT / 'hydesign/examples/thesis_simple_datacenter/simple_datacenter.yml'
defaults = read_configuration(config_path)
variables = deepcopy(defaults['variables'])

for name, lower in minimum_hpp_bounds.items():
    var = variables[name]
    upper = var['limits'][1]
    if upper <= lower:
        raise ValueError(f'Upper bound from YAML is not above lower bound for {name}: {upper} <= {lower}')
    var['var_type'] = 'design'
    var['limits'] = [lower, upper]
    if var['types'] not in ('int', 'float'):
        raise ValueError(f'Invalid variable type for {name}: {var["types"]!r}')
    if not lower <= var['value'] <= upper:
        # Reference value is not normally used to initialize native EGO, but keep it valid.
        midpoint = (lower + upper) / 2
        var['value'] = int(round(midpoint)) if var['types'] == 'int' else midpoint

# FIX IT CAPACITY, NOT WIND/PV/BESS. The model uses MW_IT and PUE to infer facility MW.
fixed_it_mw = facility_load_fixed_mw / pue
variables[IT_NAME] = dict(var_type='fixed', types='float', value=fixed_it_mw)

# Separate root; the original joint HPP–DC batch directory is untouched.
batch_root = ROOT / 'hpp_sizing_runs' / 'fixed_16mw_reliability_sweep'

config = dict(
    variables=variables,
    pue=pue,
    reliability_target=reliability_targets[0],
    datacenter_firm_revenue_per_mwh_it=revenue_rate_eur_per_mwh_it,
    joint_penalty_scaling='reference_revenue',
    joint_penalty_gamma=0.25,
    joint_penalty_mode='rational', joint_penalty_A=2.0,
    joint_penalty_lambda=0.20, joint_penalty_epsilon=0.0001,
    joint_economic_bound_factor=1.0, joint_value_scale_factor=1.0,
    ems_time_limit_seconds=60, ems_threads=1,
    ems_beta_max_reliability=0.25, ems_kappa_max_reliability=0.10,
    ems_mip_gap_max_reliability=0.0001, ems_seed_max_reliability=0,
    hpp_sizing_time_limit_seconds=outer_time_budget,
    hpp_sizing_recovery_time_limit_seconds=300,
    hpp_sizing_reporting_time_limit_seconds=120,
    hpp_sizing_prevent_sleep=True,
    n_procs=2, n_doe=16, max_iter=1000, n_clusters=2,
    npred=10000, n_seed=0, tol=1e-6, min_conv_iter=1000,
    hpp_sizing_max_unique_evaluations=None,
    hpp_sizing_output_dir=str(batch_root),
)

print(f'Facility: {facility_load_fixed_mw:.2f} MW | IT: {fixed_it_mw:.4f} MW | PUE: {pue}')
print(f'Reliability targets: {reliability_targets} | {outer_time_budget/60:g} min per target')
print(f'Native output root: {batch_root}')
print('HPP design bounds:', {name: variables[name]['limits'] for name in minimum_hpp_bounds})


## Optional fixed-HPP validation (unchanged)

The optional block below evaluates a single fixed **HPP** rather than optimizing its capacities. It remains disabled by default. For the overnight sweep, skip this block and run the next section.


In [ ]:
""" if run_fixed_validation and saved_run is None:
    fixed = deepcopy(config)
    fixed['variables'] = deepcopy(variables)
    for var in fixed['variables'].values():
        var['var_type'] = 'fixed'
    fixed_result = run_native_joint_hpp_dc_sizing(config_path, fixed)
    print(fixed_result['final_status']) """


## Overnight reliability sweep — fixed 16 MW facility

Execute the batch cell once. It runs targets **sequentially** and updates `summary.csv` after every target (even an unsuccessful one). Each run is placed in `runs/reliability_XX/`, and its `evaluations.csv` is additionally copied to the batch's own `evaluations/` subfolder. The native run folder also keeps the dispatch archives, per-evaluation checkpoints, result JSON, and figures. No modifications to HyDesign modules are necessary.

The reliability threshold is hourly: for a full 8760-hour year, the allowed failed hours are floor((1−target) × 8760). The final accepted design must satisfy the selected target. If any scenario fails, the next scenario still runs; check `error` and `run_directory` in the summary.


In [ ]:
def _target_label(target):
    return f'{100 * target:.1f}'.replace('.', 'p')


def _write_batch_summary(rows, destination):
    # Replace the CSV atomically so previous completed scenarios remain readable.
    tmp = destination.with_name(destination.name + '.tmp')
    pd.DataFrame(rows).to_csv(tmp, index=False)
    tmp.replace(destination)


if saved_run is None:
    batch_directory = (batch_root /
        ('reliability_batch_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%SZ_')
         + uuid4().hex[:8]))
    (batch_directory / 'runs').mkdir(parents=True, exist_ok=False)
    (batch_directory / 'evaluations').mkdir(parents=True, exist_ok=True)

    # Save the requested configuration, including actual HPP bounds and fixed IT MW.
    batch_manifest = dict(
        description='HPP sizing with fixed 16 MW facility at varying reliability targets',
        fixed_facility_load_mw=facility_load_fixed_mw,
        fixed_it_capacity_mw=fixed_it_mw,
        pue=pue,
        revenue_eur_per_mwh_it=revenue_rate_eur_per_mwh_it,
        reliability_targets=reliability_targets,
        time_budget_seconds_per_target=outer_time_budget,
        notebook_config=config,
    )
    (batch_directory / 'batch_configuration.json').write_text(
        json.dumps(batch_manifest, indent=2, default=str), encoding='utf-8')

    records = []
    print('Batch folder:', batch_directory, flush=True)
    for target in reliability_targets:
        label = _target_label(target)
        target_folder = batch_directory / 'runs' / f'reliability_{label}'
        scenario_config = deepcopy(config)
        scenario_config['reliability_target'] = target
        scenario_config['hpp_sizing_output_dir'] = str(target_folder)

        row = dict(
            target=target,
            target_percent=100 * target,
            allowed_failed_hours=math.floor((1.0 - target) * 8760 + 1e-9),
            fixed_facility_load_mw=facility_load_fixed_mw,
            fixed_it_capacity_mw=fixed_it_mw,
            pue=pue,
            revenue_eur_per_mwh_it=revenue_rate_eur_per_mwh_it,
            search_budget_seconds=outer_time_budget,
            status='RUNNING',
            run_directory='',
            evaluations_csv='',
            error='',
        )
        records.append(row)
        _write_batch_summary(records, batch_directory / 'summary.csv')
        print(f'\nReliability {100*target:.1f}%: starting native EGO', flush=True)

        try:
            result_item = run_native_joint_hpp_dc_sizing(config_path, scenario_config)
            run_path = Path(result_item['run_directory'])
            row['run_directory'] = str(run_path)
            row['status'] = result_item.get('final_status', 'UNKNOWN')
            row['evaluations'] = result_item.get('evaluated_designs')
            row['feasible_evaluations'] = result_item.get('feasible_evaluations')
            row['below_target_evaluations'] = result_item.get('below_target_evaluations')
            row['search_seconds'] = result_item.get('optimization_seconds')

            evaluations_path = run_path / 'evaluations.csv'
            if evaluations_path.exists():
                exported = batch_directory / 'evaluations' / f'reliability_{label}.csv'
                shutil.copy2(evaluations_path, exported)
                row['evaluations_csv'] = str(exported)

            best_item = result_item.get('best')
            if best_item:
                design = best_item['design']
                metrics = best_item['metrics']
                economics = best_item['economics']
                row.update(
                    reliability_achieved=metrics['firm_reliability'],
                    failed_hours=metrics['LOLH'],
                    eens_mwh=metrics.get('EENS'),
                    wind_MW=design[2] * design[3],
                    n_wind_turbines=design[3],
                    pv_MW_ac=design[5],
                    battery_power_MW=design[9],
                    battery_duration_h=design[10],
                    battery_energy_MWh=design[9] * design[10],
                    net_annual_value_eur=economics['net_annual_value_system'],
                    dc_revenue_eur=economics['Revenue_dc'],
                    annualized_hpp_cost_eur=economics['annualized_cost_hpp'],
                    annualized_dc_cost_eur=economics['annualized_cost_dc'],
                    hpp_lcoe_eur_per_mwh=best_item.get('lcoe_hpp_eur_per_mwh'),
                )
                print(f"  Best validated: value EUR {row['net_annual_value_eur']:,.0f}/yr; "
                      f"LOLH {row['failed_hours']}; HPP annual cost EUR "
                      f"{row['annualized_hpp_cost_eur']:,.0f}/yr", flush=True)
            else:
                print(f'  Status: {row["status"]} — no validated feasible design', flush=True)
        except Exception as exc:
            row['status'] = 'ERROR'
            row['error'] = f'{type(exc).__name__}: {exc}'
            traceback.print_exc()
        finally:
            _write_batch_summary(records, batch_directory / 'summary.csv')

    reliability_results = pd.read_csv(batch_directory / 'summary.csv')
    print('\nCombined summary:', batch_directory / 'summary.csv')
    view_cols = [
        'target_percent', 'allowed_failed_hours', 'status',
        'reliability_achieved', 'failed_hours', 'wind_MW', 'pv_MW_ac',
        'battery_power_MW', 'battery_energy_MWh', 'evaluations',
        'feasible_evaluations', 'run_directory',
    ]
    batch_view = reliability_results.reindex(columns=view_cols).copy()
    for name, label in {
        'net_annual_value_eur': 'Annual system value [MEUR/year]',
        'annualized_hpp_cost_eur': 'HPP annualized cost [MEUR/year]',
        'annualized_dc_cost_eur': 'DC annualized cost [MEUR/year]',
    }.items():
        batch_view[label] = pd.to_numeric(
            reliability_results.reindex(columns=[name])[name], errors='coerce') / 1e6
    display(batch_view.round(3))

    completed = reliability_results.loc[
        reliability_results['run_directory'].notna(), 'run_directory']
    if completed.empty:
        raise RuntimeError(f'No run result was saved. Inspect {batch_directory / "summary.csv"}')
    run_directory = Path(completed.iloc[-1])   # Detailed diagnostics below show the last saved run.
else:
    run_directory = Path(saved_run)
    batch_directory = run_directory.parent

result, history = load_joint_hpp_dc_sizing(
    run_directory, save_figures=regenerate_saved_figures)


## Results
Below-target time-limited dispatches remain in the evaluation history, with their solver certificates; they are not accepted as final designs.


In [ ]:

print('Results:', run_directory)
print('Status:', result['final_status'])
print('Evaluations:', result.get('evaluated_designs'),
      '| feasible:', result.get('feasible_evaluations'),
      '| search seconds:', result.get('optimization_seconds'))

best = result.get('best')

if best:
    design = best['design']
    pue = result['configuration']['pue']

    summary = {
        # Data center
        'IT capacity [MW]': design[11],
        'Facility demand [MW]': design[11] * pue,

        # Reliability
        'Reliability target [%]': 100 * result['configuration']['reliability_target'],
        'Achieved reliability [%]': 100 * best['metrics']['firm_reliability'],
        'LOLH [h]': best['metrics']['LOLH'],

        # HPP sizing
        'Wind [MW]': design[2] * design[3],
        'PV [MW AC]': design[5],
        'Battery power [MW]': design[9],
        'Battery duration [h]': design[10],
        'Battery energy [MWh]': design[9] * design[10],

        # Economics
        'Net annual system value [MEUR/year]':
            best['economics']['net_annual_value_system'] / 1e6,
        'DC revenue [MEUR/year]':
            best['economics']['Revenue_dc'] / 1e6,
        'Annualized HPP cost [MEUR/year]':
            best['economics']['annualized_cost_hpp'] / 1e6,
        'Annualized DC cost [MEUR/year]':
            best['economics']['annualized_cost_dc'] / 1e6,
        'Reliability penalty [MEUR/year]':
            best['scoring']['reliability_penalty_eur_per_year'] / 1e6,
        'Penalized EGO objective [MEUR/year]':
            best['scoring']['penalized_annual_value'] / 1e6,
        'HPP LCoE [EUR/MWh]': best['lcoe_hpp_eur_per_mwh'],
    }

    summary_table = pd.Series(summary, name='Validated best design').to_frame().astype(object)
    money_rows = [name for name in summary if '[MEUR/year]' in name]
    summary_table.loc[money_rows, 'Validated best design'] = summary_table.loc[
        money_rows, 'Validated best design'].map(lambda value: f'{value:,.2f}')
    display(summary_table)
    print('Penalty scaling:', result['configuration'].get('joint_penalty_scaling', 'economic_bound'))

    print('IT capacity bound:', best['it_capacity_bound'])
    print('Validation:', best['validation'])
    print('EMS status:', {k: best['solver_status'].get(k) for k in
                          ('termination', 'stages', 'certificate', 'binary_failed_hours')})
else:
    print('No validated reliability-feasible design recovered; inspect evaluations.csv.')

# Presentation only: the original history and saved EUR values stay unchanged.
annual_money = {
    'net_annual_value_system': 'Net annual system value [MEUR/year]',
    'Revenue_dc': 'DC revenue [MEUR/year]',
    'annualized_cost_hpp': 'Annualized HPP cost [MEUR/year]',
    'annualized_cost_dc': 'Annualized DC cost [MEUR/year]',
    'reliability_penalty_eur_per_year': 'Reliability penalty [MEUR/year]',
    'score': 'Penalized EGO score / fallback [MEUR/year]',
}
history_columns = ['evaluation_index', 'status', 'Nwt', 'wind_MW', 'solar_MW',
                   'b_P', 'b_E', 'it_capacity_mw', 'reliability', 'failed_hours',
                   'EENS', 'lcoe_hpp_eur_per_mwh', 'joint_penalty_scaling', 'joint_penalty_gamma']
history_view = history.reindex(columns=history_columns).copy()
for name, label in annual_money.items():
    history_view[label] = pd.to_numeric(history.reindex(columns=[name])[name], errors='coerce') / 1e6
history_display = history_view.tail().copy()
for label in annual_money.values():
    history_display[label] = history_display[label].map(
        lambda value: f'{value:,.2f}' if pd.notna(value) else '-')
display(history_display)


## Saved figures and reloading

Each native run keeps its own `configuration.json`, per-candidate JSON and dispatch archives in `evaluations/`, `evaluations.csv`, `result.json`, the selected hourly dispatch, and PNG/PDF figures. The notebook's detailed results above show the **last completed reliability scenario**; the batch `summary.csv` covers all reliability targets.

Directory structure (created within your HyDesign project, not alongside this downloaded notebook):

```text
HyDesign/hpp_sizing_runs/fixed_16mw_reliability_sweep/
  reliability_batch_<UTC timestamp>_<id>/
    summary.csv
    batch_configuration.json
    evaluations/
      reliability_90p0.csv
      reliability_95p0.csv
      reliability_99p0.csv
      reliability_99p5.csv
      reliability_99p9.csv
    runs/
      reliability_90p0/<native timestamp>_joint_hpp_dc_sizing/
      reliability_95p0/<native timestamp>_joint_hpp_dc_sizing/
      ...
```

To inspect any completed run without optimization, enter the corresponding native directory in `saved_run`, then rerun the **configuration cell** and the **batch/results cell**. Monetary values in saved CSVs remain in EUR/year; the displayed tables convert to MEUR/year.


In [ ]:
figure_folders = sorted(Path(run_directory).glob('figures_*'))
if figure_folders:
    plot = figure_folders[-1] / 'joint_hpp_dc_sizing.png'
    if plot.exists():
        display(Image(filename=str(plot)))
